Nível 1: Básico — Configuração e SQL Puro com Segurança
Nesta etapa, os alunos farão a conexão inicial e utilizarão SQL nativo, compreendendo a
importância de proteger o banco de dados contra ataques.






● Passo 1: Crie a conexão com um banco de dados local utilizando a função create_engine('sqlite:///sistema_rh.db').


In [ ]:
import os
import pandas as pd

from sqlalchemy import (Column, Float, Integer, MetaData, String, Table,
                        and_, or_, create_engine, delete, func, insert,
                        select, text, update)

# Reinicia o banco: remove o arquivo anterior, se existir
if os.path.exists('sistema_rh.db'):
    os.remove('sistema_rh.db')

# SQLite em arquivo — sem necessidade de servidor
engine = create_engine('sqlite:///sistema_rh.db')

print('Engine pronto. Banco:', engine.url.database)

Engine pronto. Banco: sistema_rh.db


● Passo 2: Abra uma transação com with engine.begin() as conn: e utilize a função
text() para executar um comando CREATE TABLE em SQL puro, criando uma tabela
chamada funcionarios (com id, nome, cargo e salario).

In [ ]:
from sqlalchemy import text

with engine.begin() as conn:
    conn.execute(text(
        """
        CREATE TABLE funcionarios (
            id INTEGER PRIMARY KEY,
            nome TEXT NOT NULL,
            cargo TEXT NOT NULL,
            salario REAL NOT NULL
        )
        """
    ))

● Passo 3 (Segurança): Simule a inserção de um novo funcionário a partir de um
formulário web. Utilize o comando INSERT passando parâmetros seguros (ex:
:nome, :cargo) e um dicionário de valores correspondentes. Pergunta reflexiva para
os alunos: Por que nunca devemos concatenar strings diretamente no SQL (risco de
injeção SQL) e como os placeholders resolvem isso?

In [ ]:
with engine.begin() as conn:
  conn.execute(text('''
  INSERT INTO funcionarios(id, nome, cargo, salario)
  VALUES (:id, :nome, :cargo, :salario)
  '''),[
      {'id': 1, 'nome': 'João', 'cargo': 'Desenvolvedor', 'salario': 5000.0},
      {'id': 2, 'nome': 'Maria', 'cargo': 'Gerente', 'salario': 7000.0}
  ])

  print('Dados inseridos com sucesso!')

Dados inseridos com sucesso!


● Passo 4: Valide a inserção consultando os dados com a função
pd.read_sql_query(), que já retorna a tabela formatada diretamente como um
DataFrame do Pandas.

In [ ]:
import pandas as pd

df_funcionario = pd.read_sql_query('SELECT * FROM funcionarios', engine)
df_funcionario

,id,nome,cargo,salario
0,1,João,Desenvolvedor,5000.0
1,2,Maria,Gerente,7000.0



Nível 2: Intermediário — SQLAlchemy Core (Automatização Programática)
O objetivo agora é abandonar o SQL em texto e usar as estruturas Python do SQLAlchemy
Core, ideais para scripts de manipulação de dados e relatórios.

● Passo 1: Defina uma nova tabela chamada projetos de forma programática
utilizando os objetos Table, MetaData e Column. Em seguida, crie a tabela
fisicamente no banco com metadata.create_all(engine).

In [ ]:

metadata = MetaData()

projetos = Table(
    'projetos', metadata,
    Column('id', Integer, primary_key=True),
    Column('nome', String(60)),
    Column('descricao', String(200)),
    Column('data_inicio', String(10)),
    Column('data_fim', String(10))
)

metadata.create_all(engine)
print('Tabela projetos criada')

Tabela projetos criada



● Passo 2: Receba uma lista de dicionários contendo múltiplos projetos e faça uma
inserção em lote (bulk insert) passando essa lista para o comando
conn.execute(insert(projetos), [lista_de_dicts]).

In [ ]:
lista_de_projetos = [
    {'id': 1, 'nome': 'Website Empresarial', 'descricao': 'Desenvolvimento de um novo site para a empresa', 'data_inicio': '2023-01-15', 'data_fim': '2023-06-30'},
    {'id': 2, 'nome': 'Aplicativo Mobile', 'descricao': 'Criação de aplicativo mobile para iOS e Android', 'data_inicio': '2023-03-01', 'data_fim': '2023-12-31'},
    {'id': 3, 'nome': 'Sistema de CRM', 'descricao': 'Implementação de um novo sistema de gerenciamento de clientes', 'data_inicio': '2023-05-20', 'data_fim': '2024-02-28'}
]

with engine.begin() as conn:
    conn.execute(insert(projetos), lista_de_projetos)

print('Dados inseridos com sucesso!')

Dados inseridos com sucesso!



● Passo 3: A diretoria aprovou um reajuste. Utilize a instrução
update(tabela).where(...).values(...) para aumentar o salário apenas dos funcionários
que ocupam o cargo de 'Desenvolvedor Júnior'.

In [ ]:

with engine.begin() as conn:
  conn.execute(update(funcionarios).where(funcionarios.c.cargo == 'Desenvolvedor').values(salario=10000))

pd.read_sql_query('SELECT * FROM funcionarios', engine)

,id,nome,cargo,salario
0,1,João,Desenvolvedor,10000.0
1,2,Maria,Gerente,7000.0


● Passo 4: Gere um relatório salarial agregando os dados. Construa um select
combinando func.avg() para calcular a média salarial e group_by() para agrupar o
resultado por cargo.

In [ ]:
stmt = select(funcionarios.c.cargo,func.avg(funcionarios.c.salario)).group_by(funcionarios.c.cargo)

pd.read_sql_query(stmt, engine)


,cargo,avg_1
0,Desenvolvedor,10000.0
1,Gerente,7000.0



Nível 3: Avançado — ORM (Orientação a Objetos e Relacionamentos)
Na fase final, a turma aplicará o padrão ORM (Object-Relational Mapping), que é
amplamente utilizado no desenvolvimento de aplicações modernas.

In [ ]:
from sqlalchemy import Column, Integer, String, ForeignKey
from sqlalchemy.orm import declarative_base, relationship

Base = declarative_base()

class Departamento(Base):
    __tablename__ = 'departamentos'
    id = Column(Integer, primary_key=True)
    nome = Column(String(60))


    funcionarios = relationship('FuncionarioORM', back_populates='departamento')

class FuncionarioORM(Base):
    __tablename__ = 'funcionarios'
    id = Column(Integer, primary_key=True)
    nome = Column(String(60))
    cargo = Column(String(60))
    salario = Column(Float)

    departamento_id = Column(Integer, ForeignKey('departamentos.id'))


    departamento = relationship('Departamento', back_populates='funcionarios')

print('Classes ORM Departamento e FuncionarioORM definidas.')


Base.metadata.create_all(engine)
print('Tabelas ORM (Departamento e FuncionarioORM) com relacionamento criadas/atualizadas.')

Classes ORM Departamento e FuncionarioORM definidas.
Tabelas ORM (Departamento e FuncionarioORM) com relacionamento criadas/atualizadas.



● Passo 1: Transforme as tabelas em classes Python. Utilize declarative_base() e
defina as classes Departamento e FuncionarioORM mapeando as colunas com
mapped_column.


● Passo 2: Estabeleça a relação entre as classes configurando uma ForeignKey na
tabela de funcionários e utilizando a função relationship em ambas as classes para
permitir a navegação de objeto para objeto.


● Passo 3: Crie uma fábrica de sessões com sessionmaker(bind=engine) e instancie
uma Session. Crie um objeto da classe Departamento e adicione funcionários a ele.
Persista todos no banco de dados de uma só vez utilizando sessao.add() e
sessao.commit().



● Passo 4: Faça uma consulta orientada a objetos para listar todos os funcionários do
departamento de "TI". Utilize sessao.execute(select(...)).scalars().all() para que o
SQLAlchemy retorne os objetos instanciados da classe em vez de linhas textuais.
No final, utilize sessao.close() para fechar a sessão adequadamente